# Run the ML batch jobs against both backends

Runs `anomaly_detection.py` (Isolation Forest), `usage_profiling.py`
(KMeans), and `forecasting.py` (Prophet) against whatever data currently
exists — mirrors the old standalone-script pattern, except now each model
lives in its own file so it can be scheduled independently in production
(e.g. anomaly detection every 4 hours via cron/systemd timer, usage
profiling every 3 months, forecasting monthly). This notebook is only for
manual/ad-hoc "run everything now" use during development.

Each backend must already exist and have its base data loaded first:

```
cd Backend  && flask db upgrade && python seed.py
cd BackendDotNet && dotnet ef database update && dotnet run -- --seed
```

A backend whose database file isn't found yet is skipped with a warning
rather than failing the whole run.

In [ ]:
import anomaly_detection
import usage_profiling
import forecasting
import billing
from common import open_backend_connections

## Run

In [ ]:
summary = {}
for backend in open_backend_connections():
    result = {
        "usage_summaries_written": billing.run(backend),
        "anomalies_flagged": anomaly_detection.run(backend),
        "usage_profiles_updated": usage_profiling.run(backend),
        "forecasts_written": forecasting.run(backend),
    }
    summary[backend.name] = result

    print(f"[{backend.name}]")
    print(f"  Billing:          {result['usage_summaries_written']} usage_summary row(s) written.")
    print(f"  Isolation Forest: {result['anomalies_flagged']} new anomaly flag(s) created.")
    print(f"  KMeans:           {result['usage_profiles_updated']} employee usage profile(s) updated.")
    print(f"  Forecasting:      {result['forecasts_written']} department forecast(s) written.")

summary